# 02 - Pipeline Health Check

Skim the last 30 days of pipeline output in 10 seconds. The notebook reads
`checkpoints/stage_data_health.json` (produced by `stage_data_health` in
`pipeline/orchestrator.py`) and renders:

1. A 1-row summary with conditional formatting (green = 0 hits, red = > 0)
2. The 50 most extreme recent entries per check (sorted by score desc)
3. A histogram of per-meter medians so the overall distribution is visible

**When to run this** - after every `convert_real_data.bat` run, before
opening the dashboard. If the summary shows > 0 in any row, open
`01_data_correction.ipynb` to investigate.

**Threshold tuning** - if every row is red, the defaults in
`pipeline/data_quality.py` are too tight for the current dataset. The
notebook does not change the threshold itself; that lives in the pipeline
code so it stays in one place.

In [ ]:
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 200)

NB_DIR = Path.cwd()
if not (NB_DIR / "_corrections_helper.py").exists():
    NB_DIR = Path(r"C:\Users\Administrator\.openclaw\workspace\portfolio\scripts\notebooks")
sys.path.insert(0, str(NB_DIR))

import _corrections_helper as h

CHECKPOINT = Path(r"C:\Users\Administrator\.openclaw\workspace\portfolio\checkpoints\stage_data_health.json")
if not CHECKPOINT.exists():
    raise SystemExit(
        f"{CHECKPOINT} not found.\n"
        "Run the pipeline first:  python -m pipeline.orchestrator"
    )
with CHECKPOINT.open("r", encoding="utf-8") as f:
    health = json.load(f)
print(f"loaded: {CHECKPOINT}")
print(f"summary: {health['summary']}")

In [ ]:
summary = health["summary"]
row = pd.DataFrame([{
    "per_meter_outliers": summary["per_meter_outliers"],
    "daily_jumps": summary["daily_jumps"],
    "negative_pairs": summary["negative_pairs"],
    "recent_window_days": summary.get("recent_window_days", 30),
    "cutoff_date": summary.get("cutoff_date", ""),
}])

# Plain-text fallback first: always print the numbers, color-coded via
# text markers ([OK] green, [WARN] red) so the user can scan in any
# rendering (Jupyter, nbconvert text, GitHub preview). The styled
# DataFrame is a bonus for interactive use, but is not required.
def _marker(n: int) -> str:
    return "OK  " if n == 0 else "WARN"

print(
    f"{_marker(summary['per_meter_outliers'])} per_meter_outliers : {summary['per_meter_outliers']:>7,}\n"
    f"{_marker(summary['daily_jumps'])} daily_jumps            : {summary['daily_jumps']:>7,}\n"
    f"{_marker(summary['negative_pairs'])} negative_pairs         : {summary['negative_pairs']:>7,}\n"
    f"     recent_window_days      : {summary.get('recent_window_days', 30):>7,}\n"
    f"     cutoff_date             : {summary.get('cutoff_date', ''):>7}"
)

# Optional styled table (requires jinja2, which ships with newer pandas
# installs but not all minimal envs). Skip silently if missing.
try:
    def _color_zero(v):
        if v == 0:
            return "background-color: #c8e6c9; color: #1b5e20"
        if v > 0:
            return "background-color: #ffcdd2; color: #b71c1c"
        return ""

    styled = row.style.applymap(
        _color_zero,
        subset=["per_meter_outliers", "daily_jumps", "negative_pairs"],
    ).format({
        "per_meter_outliers": "{:,}",
        "daily_jumps": "{:,}",
        "negative_pairs": "{:,}",
        "recent_window_days": "{:,}",
    })
    display(styled)
except (ImportError, AttributeError, ValueError) as e:
    print(f"\n(styled table skipped: {type(e).__name__}: {e})")

print()
n_total = (
    summary["per_meter_outliers"]
    + summary["daily_jumps"]
    + summary["negative_pairs"]
)
if n_total == 0:
    print("OK  no anomalies detected in any check - clean run.")
else:
    print(
        f"WARN  {n_total:,} total anomalies across the three checks.\n"
        f"      Open 01_data_correction.ipynb to investigate the most extreme entries."
    )

In [ ]:
rows = health.get("recent_per_meter_outliers", [])
print(f"recent_per_meter_outliers: {len(rows)} entries (top 50 from last 30 days)")
if rows:
    df = pd.DataFrame(rows)
    df = df[["date", "meterId", "value", "score"]]
    print(df.to_string(index=False))
else:
    print("(none - no per-meter z > 4 spikes in the recent window)")

In [ ]:
rows = health.get("recent_daily_jumps", [])
print(f"recent_daily_jumps: {len(rows)} entries (top 50 from last 30 days)")
if rows:
    df = pd.DataFrame(rows)
    df = df[["date", "meterId", "value", "score"]]
    print(df.to_string(index=False))
else:
    print("(none - no 20x+ jumps in the recent window)")

In [ ]:
rows = health.get("recent_negative_pairs", [])
print(f"recent_negative_pairs: {len(rows)} entries (top 50 from last 30 days)")
if rows:
    df = pd.DataFrame(rows)
    df = df[["date", "meterId", "value", "score"]]
    print(df.to_string(index=False))
else:
    print("(none - no cancellation-style entries in the recent window)")

In [ ]:
df = h.load_cache_as_df()
if df.empty:
    print("cache is empty - run convert_real_data.bat first")
else:
    medians = df.groupby("meterId")["total"].median()
    nonzero = medians[medians > 0]
    print(
        f"{len(nonzero):,} meters with non-zero median  |  "
        f"min={nonzero.min():.1f}  p25={nonzero.quantile(.25):.1f}  "
        f"median={nonzero.median():.1f}  p75={nonzero.quantile(.75):.1f}  "
        f"p99={nonzero.quantile(.99):.1f}  max={nonzero.max():.1f}"
    )
    bins = np.logspace(np.log10(max(1, nonzero.min())), np.log10(nonzero.max()), 25)
    counts, edges = np.histogram(nonzero, bins=bins)
    print()
    print("Per-meter median (log-spaced bins):")
    print(f"{'bin_low':>10}  {'bin_high':>10}  {'count':>7}  bar")
    for i, c in enumerate(counts):
        bar = "#" * int(40 * c / max(1, counts.max()))
        print(f"{edges[i]:>10.1f}  {edges[i+1]:>10.1f}  {c:>7,}  {bar}")

## Notes

- This notebook is **read-only** with respect to the data and the
  corrections file. If the summary shows entries that need fixing,
  open `01_data_correction.ipynb` and follow the investigate - apply
  - verify flow there.
- The `recent_*` lists are capped at 50 entries per check; the full
  lists are in `*_all` in the same JSON for any deep dive.
- The histogram is **log-binned** so a small number of large
  consumers (hotels, casinos) don't squash the long tail of small
  residential meters. Look for an unusually heavy right tail as a
  sanity check on the recent spike.
- The `recent_window_days` field in the summary reflects the
  pipeline's `cutoff = max(date) - 30 days`. If the most recent
  data in the cache is older than 30 days, the recent list may be
  empty even though the all-time list is not.